### System Assumptions

For this project the following is assumed

1. Alice and Bob each have a long-term signing key pair.
2. Alice already knows Bob’s authentic public signing key.
3. Bob already knows Alice’s authentic public signing key.
4. This public-key trust setup happens before the protocol begins.

These assumptions are established in the following cells

In [87]:
from source.user import User

In [88]:
# Creating the users
alice = User("Alice")
bob = User("Bob")

# Generating their key pairs with RSA
alice.generate_key_pair()
bob.generate_key_pair()

# Give Alice and Bob's public keys to eachother
alice.set_peer_identity_public_key(bob.identity_public_key)
bob.set_peer_identity_public_key(alice.identity_public_key)

Generated identity key pair for Alice
Generated identity key pair for Bob
Set peer identity key pair for Alice
Set peer identity key pair for Bob


Creating a session between Alice and Bob, setting a session id, however we still need to do the DH key exchange before Alice and Bob can start exchanging messages.

In [89]:
import os

shared_session_id = os.urandom(8)

alice.start_session(shared_session_id, bob.name)
bob.start_session(shared_session_id, alice.name)

### Diffie-Hellman Key Exchange

The next step is doing the DH key exchange. It should be done in such a way that the receiver can verify that the key truly came from the sender by making sure the passed dh public key matches with the one signed with the digital signature

In [90]:
# Alice and Bob privately create their DH key pairs
alice.generate_dh_key_pair()
bob.generate_dh_key_pair()

# Now Alice and Bob send eachother their DH public key, along with their signature for authentication
alice_dh_public_key, alice_signature = alice.sign_dh_with_identity_private_key()
bob_dh_public_key, bob_signature = bob.sign_dh_with_identity_private_key()

# --------------------------------
# INSECURE, ATTACKER-CONTROLLED CHANNEL
# The attacker can inspect or modify values here
# --------------------------------

# Alice privately verifies Bob's DH public key using the provided signature
if alice.verify_signature(bob_dh_public_key, bob_signature):
    # If the signature is valid then accept his DH public key
    alice.set_peer_dh_public_key(bob_dh_public_key)
else:
    raise ValueError("Alice rejected Bob's DH public key.")
    

# Bob privately verifies Alice's DH public key using the provided signature
if bob.verify_signature(alice_dh_public_key, alice_signature):
    # If the signature is valid then accept her DH public key
    bob.set_peer_dh_public_key(alice_dh_public_key)
else:
    raise ValueError("Bob rejected Alice's DH public key.")


Generated emphemeral DH key pair for Alice
Generated emphemeral DH key pair for Bob
Signature and Content Match
DH public key set for Alice
Signature and Content Match
DH public key set for Bob


### Shared Session Key

The next step is to compute the shared session key and use it to derive sending and receiving keys for each user for the session.

In [91]:
# Now Alice and Bob privately can compute the shared session secret
alice.set_shared_session_secret()
bob.set_shared_session_secret()

# After computing the session secret, they each privately create their own sending and receiving keys
alice.set_session_sending_receiving_key(
    sending_info=b"secure-messaging:alice-to-bob",
    receiving_info=b"secure-messaging:bob-to-alice",
)

bob.set_session_sending_receiving_key(
    sending_info=b"secure-messaging:bob-to-alice",
    receiving_info=b"secure-messaging:alice-to-bob",
)

Shared session secret set for Alice
Shared session secret set for Bob
Session sending and receiving keys set for Alice
Session sending and receiving keys set for Bob


### Sending and Receiving Messages

At this point we can assume that the session has been established, so Alice and Bob can start sending and receiving messages.

Messages are encrypted using AES GCM, a random nonce, and a sequence number being tracked for each participant

In [92]:
# Alice encrypts a set of messages
alice_packet_1 = alice.encrypt_message("Hi Bob.")
alice_packet_2 = alice.encrypt_message("How are you?")
alice_packet_3 = alice.encrypt_message("I think someone can see our messages.")

# Bob receives and decrypts those messages
bob.decrypt_message(alice_packet_1)
bob.decrypt_message(alice_packet_2)
bob.decrypt_message(alice_packet_3)

# Bob inspects the result
bob.received_session_messages

['Hi Bob.', 'How are you?', 'I think someone can see our messages.']

In [93]:
# Bob encrypts a set of messages
bob_packet_1 = bob.encrypt_message("Hi Alice, I'm good and you?")
bob_packet_2 = bob.encrypt_message("Yes I got that sense as well.")
bob_packet_3 = bob.encrypt_message("But I'm sure the developer made sure this was secure.")

# Alice receives and decrypts that message
alice.decrypt_message(bob_packet_1)
alice.decrypt_message(bob_packet_2)
alice.decrypt_message(bob_packet_3)

# Alice inspects the result
alice.received_session_messages

["Hi Alice, I'm good and you?",
 'Yes I got that sense as well.',
 "But I'm sure the developer made sure this was secure."]

### Attack Demonstration

We'll make a new session and pose as the attacker, seeing what happens in response to a number of different attacks. We'll start with the initial session setup below

In [94]:
import os

shared_session_id = os.urandom(8)

# Cleaning up the last session
alice.delete_session()
bob.delete_session()

# Making a new session
alice.start_session(shared_session_id, bob.name)
bob.start_session(shared_session_id, alice.name)

# Message lists should be empty
print(alice.received_session_messages)
print(bob.received_session_messages)

# Give Alice and Bob's public keys to eachother
alice.set_peer_identity_public_key(bob.identity_public_key)
bob.set_peer_identity_public_key(alice.identity_public_key)

# Alice and Bob privately create their DH key pairs
alice.generate_dh_key_pair()
bob.generate_dh_key_pair()

# Now Alice and Bob create their DH public key, along with their signature for authentication
alice_dh_public_key, alice_signature = alice.sign_dh_with_identity_private_key()
bob_dh_public_key, bob_signature = bob.sign_dh_with_identity_private_key()

[]
[]
Set peer identity key pair for Alice
Set peer identity key pair for Bob
Generated emphemeral DH key pair for Alice
Generated emphemeral DH key pair for Bob


### Man-in-the-middle

At this point, Alice and Bob will be giving eachother their peer DH public key. In transit, an attacker may intercept the packet, create their own DH key pair, and Alice will think she's communicating with Bob but in reality she's communicating with the attacker.

The attacker intercepts Bob's packet, replaces Bob's DH public key with their own, and forwards Bob's original signature unchanged. However, Bob's signature was generated over content that included Bob's real DH public key, and when Alice verifies the signature, the result doesn't match since the signature was computed using Bob's original DH public key which of course differs from Evil Bob's.

As a result, Alice rejects the attacker's substituted DH public key, preventing the attacker from establishing a shared secret with her.

In [95]:
# In this scenario, Bob knows the session ID, and will attempt to get Alice to think he's the real Bob
evil_bob = User("Evil Bob")

# Evil Bob masquerades as Bob
evil_bob.name = "Bob"
evil_bob.start_session(shared_session_id, alice.name)

# Need a private key to generate a DH key pair
evil_bob.generate_key_pair()
evil_bob.generate_dh_key_pair()

# In theory to this point Evil Bob will have replicated the correct header, up until the DH public key
evil_bob_dh_public_key, evil_bob_signature = evil_bob.sign_dh_with_identity_private_key()

Generated identity key pair for Bob
Generated emphemeral DH key pair for Bob


In [96]:
# --------------------------------
# ATTACKER INTERCEPTS PACKET
# --------------------------------

# Steals Bob's signature
evil_bob_signature = bob_signature

# Alice then received Evil Bob's DH public key and the real Bob's signature
if alice.verify_signature(evil_bob_dh_public_key, evil_bob_signature):
    # If the signature is valid then accept his DH public key
    alice.set_peer_dh_public_key(evil_bob_dh_public_key)
    print("FAIL: The Attacker has inserted their public DH key")
else:
    # The key is rejected, as evil_bob_dh_public_key does not match what is in Bob's original signature, his authentic DH public key
    print("PASS: Evil Bob's DH public key was rejected")


PASS: Evil Bob's DH public key was rejected


Lets proceed with the normal flow to test the next attacks.

In [97]:
# Alice privately verifies Bob's DH public key using the provided signature
if alice.verify_signature(bob_dh_public_key, bob_signature):
    # If the signature is valid then accept his DH public key
    alice.set_peer_dh_public_key(bob_dh_public_key)
else:
    raise ValueError("Alice rejected Bob's DH public key.")
    

# Bob privately verifies Alice's DH public key using the provided signature
if bob.verify_signature(alice_dh_public_key, alice_signature):
    # If the signature is valid then accept her DH public key
    bob.set_peer_dh_public_key(alice_dh_public_key)
else:
    raise ValueError("Bob rejected Alice's DH public key.")

# Now Alice and Bob privately can compute the shared session secret
alice.set_shared_session_secret()
bob.set_shared_session_secret()

# After computing the session secret, they each privately create their own sending and receiving keys
alice.set_session_sending_receiving_key(
    sending_info=b"secure-messaging:alice-to-bob",
    receiving_info=b"secure-messaging:bob-to-alice",
)

bob.set_session_sending_receiving_key(
    sending_info=b"secure-messaging:bob-to-alice",
    receiving_info=b"secure-messaging:alice-to-bob",
)

Signature and Content Match
DH public key set for Alice
Signature and Content Match
DH public key set for Bob
Shared session secret set for Alice
Shared session secret set for Bob
Session sending and receiving keys set for Alice
Session sending and receiving keys set for Bob


### Header modification attack

Header modification attack assumes the attacker intercepts a message, modifies some value in the header and allows the packet to continue on to the intended recipient.

In this case, let's say the nonce was modified. We'll also do the sequence.

When the attacker sends it on, and Bob goes to decrypt it, authentication fails because the packet was modified. This happens because the nonce no longer matches the original value from when the ciphertext was originally generated. AES-GCM detects this and raises an exception.


In [98]:
# Alice creates and encrypts her packet
alice_packet_1 = alice.encrypt_message("Hi Bob.")

# --------------------------------
# ATTACKER INTERCEPTS PACKET
# --------------------------------

original_nonce = alice_packet_1["nonce"]

# The header is changed
alice_packet_1["nonce"] = os.urandom(12)

# Bob goes to decrypt the packet, and there's an invalid tag error
try:
    bob.decrypt_message(alice_packet_1)
    print("FAIL: Modified packet was accepted.")
except ValueError as error:
    print("PASS: Modified packet was rejected.")
    print("Reason:", error)

# Setting nonce back to original
alice_packet_1["nonce"] = original_nonce

original_sequence = alice_packet_1["sequence"]

# The header is changed
alice_packet_1["sequence"] = 500

# Bob goes to decrypt the packet, and there's an invalid tag error
try:
    bob.decrypt_message(alice_packet_1)
    print("FAIL: Modified packet was accepted.")
except ValueError as error:
    print("PASS: Modified packet was rejected.")
    print("Reason:", error)

# Setting sequence back to original
alice_packet_1["sequence"] = original_sequence


PASS: Modified packet was rejected.
Reason: Message authentication failed: packet was modified.
PASS: Modified packet was rejected.
Reason: Message authentication failed: packet was modified.


### Message modification attack

This attack is when the attacker intercepts the packet, changes some value in the ciphertext and the packet continues to the receiver.

It fails as AES-GCM initally computes an authentication tag when the message is encrypted, and that tag is recomputed when the message is received. The expected tag does not match the received, modified tag which throws an error and the message is rejected.

In [99]:
# Storing the correct ciphertext
original_ciphertext = alice_packet_1["ciphertext"]

# Flip the second bit of the second byte
ciphertext = bytearray(alice_packet_1["ciphertext"])
ciphertext[1] ^= 1 << 1

alice_packet_1["ciphertext"] = bytes(ciphertext)

# The message is rejected as the newly computed authentication tag does not match the authentication tag computed at encryption
try:
    bob.decrypt_message(alice_packet_1)
    print("FAIL: Modified packet was accepted.")
except ValueError as error:
    print("PASS: Modified packet was rejected.")
    print("Reason:", error)

PASS: Modified packet was rejected.
Reason: Message authentication failed: packet was modified.


Let's set the ciphertext back to what it was and accept the message so we can move on to the replay attack.

In [100]:
# Setting ciphertext back to original
alice_packet_1["ciphertext"] = original_ciphertext

# Decrypting the valid message
bob.decrypt_message(alice_packet_1)

bob.received_session_messages

['Hi Bob.']

### Replay attack

Bob has received the message successfully, however let's assume that a duplicate message is sent again.

This would be a replay attack, rejected though because the sequence in the header won't have changed, but the receiver's receive counter will have increased, with that mismatch resulting in an error and the message being rejected.

In [101]:

# -- Duplicate message is sent to the recipient --

# As shown in the error, there is a mismatch between sequence counter and recipient receive counter resulting in the rejection of the message
try:
    bob.decrypt_message(alice_packet_1)
    print("FAIL: Modified packet was accepted.")
except ValueError as error:
    print("PASS: Modified packet was rejected.")
    print("Reason:", error)

PASS: Modified packet was rejected.
Reason: Sent sequence and receive counter not matching, counter is 1, actually received 0
